In [20]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate

In [21]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sandeep\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU
PyTorch version: 2.2.2+cu121


In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
import torch
print("Torch version:", torch.__version__)
print("Torch location:", torch.__file__)

import transformers
print("Transformers version:", transformers.__version__)

Torch version: 2.2.2+cu121
Torch location: c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\torch\__init__.py
Transformers version: 4.40.0


### Dataset load

In [6]:
# Windows example
dataset = load_dataset("csv", data_files={
    "train": r"C:\Users\Sandeep\Desktop\Projects\datasets\samsum-train.csv",
    "test": r"C:\Users\Sandeep\Desktop\Projects\datasets\samsum-test.csv",
    "validation": r"C:\Users\Sandeep\Desktop\Projects\datasets\samsum-validation.csv"
})

In [7]:
# Number of rows in each split
print("Train rows:", dataset["train"].num_rows)
print("Test rows:", dataset["test"].num_rows)
print("Validation rows:", dataset["validation"].num_rows)

# Column names
print("Columns:", dataset["train"].column_names)

Train rows: 14732
Test rows: 819
Validation rows: 818
Columns: ['id', 'dialogue', 'summary']


### Load tokeniser

In [8]:
model_checkpoint = "t5-small"   # we'll start with this, upgrade later
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
tokenizer

T5TokenizerFast(name_or_path='t5-small', vocab_size=32100, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42>', '<extra_id_43>', '<extra_i

In [10]:
sample = dataset["train"][0]

inputs = tokenizer(sample["dialogue"], truncation=True, max_length=512)
target = tokenizer(sample["summary"], truncation=True, max_length=128)

print("Input tokens:", len(inputs["input_ids"]))
print("Target tokens:", len(target["input_ids"]))
print("Sample dialogue:", sample["dialogue"][:200])
print("Sample summary:", sample["summary"])

Input tokens: 28
Target tokens: 11
Sample dialogue: Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)
Sample summary: Amanda baked cookies and will bring Jerry some tomorrow.


In [11]:
# Filter out None values before tokenising
dialogues = [x for x in dataset["train"]["dialogue"] if isinstance(x, str)]
summaries = [x for x in dataset["train"]["summary"] if isinstance(x, str)]

# For T5, use with_prefix approach or just decode directly
dialogue_lengths = [len(tokenizer.encode(x)) for x in dialogues]
summary_lengths  = [len(tokenizer.encode(x)) for x in summaries]

print("Dialogue — max:", max(dialogue_lengths), "| avg:", int(np.mean(dialogue_lengths)))
print("Summary  — max:", max(summary_lengths),  "| avg:", int(np.mean(summary_lengths)))

Token indices sequence length is longer than the specified maximum sequence length for this model (567 > 512). Running this sequence through the model will result in indexing errors


Dialogue — max: 1153 | avg: 148
Summary  — max: 94 | avg: 28


In [12]:
# Find the null values
total = len(dataset["train"]["dialogue"])
null_d = sum(1 for x in dataset["train"]["dialogue"] if x is None)
null_s = sum(1 for x in dataset["train"]["summary"] if x is None)
empty_d = sum(1 for x in dataset["train"]["dialogue"] if x == "")
empty_s = sum(1 for x in dataset["train"]["summary"] if x == "")

print(f"Total rows:        {total}")
print(f"Null dialogues:    {null_d} ({null_d/total*100:.1f}%)")
print(f"Null summaries:    {null_s} ({null_s/total*100:.1f}%)")
print(f"Empty dialogues:   {empty_d}")
print(f"Empty summaries:   {empty_s}")

# Peek at a few null rows to understand why
null_indices = [i for i, x in enumerate(dataset["train"]["dialogue"]) if x is None]
print("\nSample null rows:")
for i in null_indices[:5]:
    print(dataset["train"][i])

Total rows:        14732
Null dialogues:    1 (0.0%)
Null summaries:    0 (0.0%)
Empty dialogues:   0
Empty summaries:   0

Sample null rows:
{'id': '13828807', 'dialogue': None, 'summary': 'problem with visualization of the content'}


In [13]:
# Remove the null values
dataset = dataset.filter(lambda x: x["dialogue"] is not None and x["summary"] is not None)

# Confirm
print("Train rows after cleaning:", dataset["train"].num_rows)

Train rows after cleaning: 14731


In [14]:
# TOkenizing the complete dataset
max_input_length = 512
max_target_length = 128
prefix = "summarize: "

def preprocess(examples):
    inputs = [prefix + doc for doc in examples["dialogue"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply to all splits at once
tokenized_dataset = dataset.map(preprocess, batched=True)
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})


### Load the model

In [15]:
# Using transfer learning.... loading t5 model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
model = model.to(device)
print("Model loaded!", model)

Model loaded! T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (

#### ROUGE stands for Recall-Oriented Understudy for Gisting Evaluation — it's the standard way to measure how good a summary is.

In [16]:
# Trainig arguments and rouge
rouge = evaluate.load("rouge")
print("ROUGE loaded!")

ROUGE loaded!


In [ ]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-samsum",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
)
print("Training args ready!")

Training args ready!


In [23]:
# Computing mertics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in labels (padding tokens)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Clean up whitespace — no nltk needed!
    decoded_preds  = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v, 4) for k, v in result.items()}

In [24]:
# Start training
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

 33%|███▎      | 1842/5526 [1:36:18<3:12:36,  3.14s/it]
                                                    


                                        

  2%|▏         | 100/5526 [01:43<1:31:12,  1.01s/it] 




{'loss': 0.4383, 'grad_norm': 2.448725938796997, 'learning_rate': 1e-05, 'epoch': 0.05}


                                                    


                                        

  4%|▎         | 200/5526 [03:26<1:27:37,  1.01it/s]   




{'loss': 0.4381, 'grad_norm': 0.8752299547195435, 'learning_rate': 2e-05, 'epoch': 0.11}


                                                    


                                        

  5%|▌         | 300/5526 [05:07<1:25:21,  1.02it/s]   




...

100%|██████████| 5526/5526 [2:33:54<00:00,  2.72s/it]c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\transformers\generation\utils.py:1141: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(



                                                     

                                                 
                                              

100%|██████████| 5526/5526 [2:35:27<00:00,  2.72s/it]


{'eval_loss': 0.3804456889629364, 'eval_rouge1': 0.4345, 'eval_rouge2': 0.2084, 'eval_rougeL': 0.3683, 'eval_rougeLsum': 0.3684, 'eval_runtime': 92.4636, 'eval_samples_per_second': 8.847, 'eval_steps_per_second': 1.114, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



                                                     

                                                 

100%|██████████| 5526/5526 [2:35:28<00:00,  1.69s/it]

{'train_runtime': 9328.795, 'train_samples_per_second': 4.737, 'train_steps_per_second': 0.592, 'train_loss': 0.4019772408451242, 'epoch': 3.0}


TrainOutput(global_step=5526, training_loss=0.4019772408451242, metrics={'train_runtime': 9328.795, 'train_samples_per_second': 4.737, 'train_steps_per_second': 0.592, 'total_flos': 5981160232452096.0, 'train_loss': 0.4019772408451242, 'epoch': 3.0})

In [25]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\transformers\generation\utils.py:1141: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 103/103 [01:30<00:00,  1.14it/s]

{'eval_loss': 0.3793216943740845, 'eval_rouge1': 0.4272, 'eval_rouge2': 0.1931, 'eval_rougeL': 0.3559, 'eval_rougeLsum': 0.3561, 'eval_runtime': 91.0881, 'eval_samples_per_second': 8.991, 'eval_steps_per_second': 1.131, 'epoch': 3.0}


In [ ]:
dialogue = """
John: did you watch the game last night?
Sarah: yes! it was incredible
John: that last minute goal was unbelievable
Sarah: I know! best game of the season for sure
"""

input_text = "summarize: " + dialogue
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=128,
    min_length=20,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Generated summary:", summary)

Generated summary: John watched the game last night. The last minute goal was incredible. Sarah knows the best game of the season.


In [29]:
# Save model and tokenizer
model.save_pretrained("./t5-samsum-final")

In [30]:
tokenizer.save_pretrained("./t5-samsum-final")

('./t5-samsum-final\\tokenizer_config.json',
 './t5-samsum-final\\special_tokens_map.json',
 './t5-samsum-final\\spiece.model',
 './t5-samsum-final\\added_tokens.json',
 './t5-samsum-final\\tokenizer.json')